# Causal analysis on `breast_cancer`

The causal layer of the harness, run on the **same dataset and classifier** as
`discovery_example.ipynb`. A causal claim here is a `do()` estimand with a
declared acceptance region and a declared assumption set; a capability is an
**identification strategy**; a **placebo is the twin**.

Two tiers:

| tier | question | strategy |
|---|---|---|
| **2 — cross-sectional** | does `do(x)` change `y`, and can it be identified? | backdoor adjustment, paired potential outcomes, conditional independence |
| **1 — interrupted time series** | did the intervention change the process? | segmented regression over the journal; pre-trend + placebo-date claims |

The extended write-up (concepts, assumptions, non-claims) is
`causal_analysis.md` in this directory.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # the mjolnir_ml package

import random
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from mjolnir_ml import (ApprovalAuthority, CausalClaim, CausalSession, Journal,
                        Strategy, do, series_from_journal)
from mjolnir_ml.discovery import CERTIFIED, DECLINED, REFUTED

data = load_breast_cancer()
X, y, names = data.data, data.target, list(data.feature_names)
FC = names.index("worst concave points")
FR = names.index("worst radius")
FT = names.index("mean texture")
FS = names.index("mean symmetry")
FF = names.index("mean fractal dimension")
FSM = names.index("mean smoothness")

X_train, X_audit, y_train, y_audit = train_test_split(
    X, y, test_size=0.45, random_state=0, stratify=y)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(
    X_train[:, [FC, FR]], y_train)
malignant_call = lambda A: (tree.predict(A) == 0).astype(float)
base = malignant_call(X_audit[:, [FC, FR]])
median_concave = float(np.median(X_audit[:, FC]))
p_benign = tree.predict_proba(X[:, [FC, FR]])[:, 1]
authority = ApprovalAuthority(b"bc-demo")

print(f"rows {len(y)} | train {len(y_train)} | audit {len(y_audit)}")
print(f"classifier leaves: {sorted(int(v) for v in set(tree.apply(X_audit[:, [FC, FR]])))}")
print(f"audit malignant-call rate: {base.mean():.3f}")

rows 569 | train 312 | audit 257
classifier leaves: [3, 4, 6, 7, 10, 11, 13, 14]
audit malignant-call rate: 0.354


## The classifier as an SCM, and the limits of the data

The intervention we can identify is on the **model**: `do(radius = 17.2)` sets a
feature for every audit row and re-scores it. That is a well-posed causal
question about the deployed classifier.

The intervention we **cannot** identify is on the **patient**: concavity is a
manifestation of the tumor, not a cause of it, so `do(concave)` on a patient is
not well posed. The registry keeps that claim in the frontier and no capability
targets it.

## The causal chain, in plain text

Four compact diagrams, drawn with [`phart`](https://github.com/scottvr/phart)
(pure Python, boxed Unicode output — no browser, no JavaScript):

1. **the intervention chain** — `do(...)` sets a model input; the two inputs
   meet at the prediction.
2. **the decision chain** — the prediction is gated by the declared policy
   threshold `τ`, and the outcome is the hazard-release rate defined by an
   **external** hazard rule (not by the model).
3. **the structure the etiology claim would need** — latent disease → features →
   diagnosis. The diagnosis is a **collider**: conditioning on it opens a path
   between texture and smoothness. There is deliberately **no arrow from an
   intervention into the diagnosis**: `do(concave)` on a patient is exactly the
   claim the notebook declines as unidentified.
4. **the placebo lever** — `mean symmetry` is a box with no edges: the model
   never reads it, which is why an intervention on it must have exactly zero
   effect.

Install the renderer for the notebook with `pip install phart`; the harness
itself never imports it.

In [2]:
import networkx as nx
from phart import ASCIIRenderer, LayoutOptions, NodeStyle


def render(nodes, edges):
    graph = nx.DiGraph()
    for node, label in nodes.items():
        graph.add_node(node, label=label)
    graph.add_edges_from(edges)
    options = LayoutOptions(layout_strategy="hierarchical", bboxes=True, hpad=2,
                            edge_anchor_mode="center", layer_spacing=4,
                            node_spacing=4, node_style=NodeStyle.MINIMAL,
                            use_labels=True, use_ascii=False)
    return ASCIIRenderer(graph, options=options).render()


INTERVENTION_CHAIN = (
    {"DO_R": "do(radius=17.2)", "DO_C": "do(concave=.155)", "R": "radius",
     "C": "concave", "P": "prediction"},
    [("DO_R", "R"), ("DO_C", "C"), ("R", "P"), ("C", "P")],
)
DECISION_CHAIN = (
    {"P": "prediction", "REL": "release? (policy τ)",
     "HR": "hazard rate (external rule)"},
    [("P", "REL"), ("REL", "HR")],
)
COLLIDER = (
    {"D": "latent disease", "T": "texture", "S": "smoothness", "M": "diagnosis"},
    [("D", "T"), ("D", "S"), ("T", "M"), ("S", "M")],
)
PLACEBO = ({"SYM": "mean symmetry"}, [])

print("(1) the intervention chain")
print(render(*INTERVENTION_CHAIN))
print("\n(2) the decision chain")
print(render(*DECISION_CHAIN))
print("\n(3) the collider the etiology claim would need")
print(render(*COLLIDER))
print("\n(4) the placebo lever (no edge into the prediction)")
print(render(*PLACEBO))

(1) the intervention chain
┌────────────────────┐    ┌───────────────────┐
│  do(concave=.155)  │    │  do(radius=17.2)  │
└────────────────────┘    └───────────────────┘
           └───┐                ┌───┘
               │                │
               v                v
         ┌───────────┐    ┌──────────┐
         │  concave  │    │  radius  │
         └───────────┘    └──────────┘
               │       ┌────────┘
               └───────┤
                       v
               ┌──────────────┐
               │  prediction  │
               └──────────────┘



(2) the decision chain
        ┌──────────────┐
        │  prediction  │
        └──────────────┘
                │
                │
                v
    ┌───────────────────────┐
    │  release? (policy τ)  │
    └───────────────────────┘
                │
                │
                v
┌───────────────────────────────┐
│  hazard rate (external rule)  │
└───────────────────────────────┘



(3) the collider the e

## 1. The record: association is certified, etiology is not identified

The registry is audited once with `Strategy("none")`. Associations are estimated;
the etiological `ATE` is declined as `unidentified:no-strategy`.

In [3]:
raw = [{"concave": float(X_audit[i, FC]), "radius": float(X_audit[i, FR]),
        "malignant": float(y_audit[i] == 0),
        "concave_high": float(X_audit[i, FC] >= median_concave)}
       for i in range(len(y_audit))]

assoc = CausalClaim(kind="association", outcome="malignant",
                    treatment="concave_high",
                    arms=(do("concave_high", 1), do("concave_high", 0)),
                    criterion={"min": 0.6, "max": 0.8}, label="association")
etiology = CausalClaim(kind="ate", outcome="malignant",
                       treatment="concave_high",
                       arms=(do("concave_high", 1), do("concave_high", 0)),
                       criterion={"min": 0.6, "max": 0.8},
                       assumptions=("concavity is not a descendant of "
                                    "malignancy",), label="etiology")

session = CausalSession([assoc, etiology], table=raw,
                        approval_verifier=authority.verify)
for claim in (assoc, etiology):
    print(f"{session.corpus[claim.id]:9s} {claim.id}")
    print(f"          reason: {session._reasons[claim.id]}")
print("\ncensus  :", session.status()["census"])
print("frontier:", session.identification_frontier())

certified E[malignant|concave_high=1] − E[malignant|concave_high=0] ∈ [0.6, 0.8]
          reason: certified
declined  ATE[malignant|do(concave_high=1), do(concave_high=0)] ∈ [0.6, 0.8]
          reason: unidentified:no-strategy

census  : {'certified': 1, 'declared': 0, 'declined': 1, 'refuted': 0}
frontier: {'unidentified:no-strategy': 1}


The association is ≈ 0.70 and certifies (the observed difference is real). The
causal claim is **declined, not refuted**: this data cannot answer it. Declining
is the honest verdict — refuting it would confuse "unidentified" with "false".

## 2. Over-adjustment manufactures a refutation, and containment catches it

A plausible-looking capability adjusts for `worst radius` — another
manifestation of the same disease. It reports 0.35, refuting the claim. A new
`REFUTED` is a defect signal, so the admission fails and the session contains
itself. Lifting containment needs an external signature.

In [4]:
bad = Strategy("backdoor", adjust=("radius",))
oracle_bad = session.strategy(bad)
print("over-adjusted:", oracle_bad(etiology.id), oracle_bad.reasons[etiology.id],
      oracle_bad.estimate(etiology.id))

candidate_bad = session.candidate(strategy=bad, targets=[etiology])
admission_bad = session.promote(candidate_bad, session.strategy(bad))
print("\nadmission:", admission_bad.approved, "-", admission_bad.reason)

for d in session.tick():
    print(f"  {d.name:18s} alarm={str(d.alarm):5s} {d.severity:8s} {d.detail}")
print("  auto_act ->", [c.reason for c in session.auto_act(session.tick())])

try:
    session.uncontain(authority.sign("wrong"))
except PermissionError as ex:
    print("\nunsigned lift refused:", str(ex)[:64], "...")
session.uncontain(authority.sign(session.containment.change_digest))
print("lifted with a signature; tick:", [(d.name, d.alarm) for d in session.tick()])

over-adjusted: refuted refuted Estimate(0.3496 ∈ [0.2609, 0.4382], n=257, regression-adjustment(backdoor({radius})))

admission: False - gate failed: labelled_tests, twin_holds, no_new_refuted, certified_gained, targeted_decline_moved
  new_refuted        alarm=True  critical 1 evaluation(s) with after.refuted > before.refuted
  self_certification alarm=False info     0 self-certification attempt(s)
  decline_growth     alarm=False info     0 evaluation(s) with after.declined > before.declined
  auto_act -> ['1 evaluation(s) with after.refuted > before.refuted']

unsigned lift refused: uncontain requires a valid external approval for sha256:2f60df39 ...
lifted with a signature; tick: [('new_refuted', False), ('self_certification', False), ('decline_growth', False)]


## 3. Mechanistic ATE: `do(radius = 17.2)` on the classifier

Each audit row is scored under both assignments; the **paired** strategy takes
the mean within-pair difference, which is the correct analysis for a
potential-outcomes table where every unit appears in both arms.

The placebo is a **negative control**: `mean symmetry` is not an input of the
model, so its contrast is exactly 0 for every pair — a CI of `[0, 0]`.

In [5]:
r_mid, c_mid = 17.2, 0.155
do_radius = malignant_call(np.column_stack([X_audit[:, FC],
                                            np.full(len(y_audit), r_mid)]))
do_concave = malignant_call(np.column_stack([np.full(len(y_audit), c_mid),
                                             X_audit[:, FR]]))

potential = []
for i in range(len(base)):
    potential.append({"pair": i, "set_radius": 0, "pred_radius": float(base[i]),
                      "set_concave": 0, "pred_concave": float(base[i]),
                      "set_symmetry": 0, "pred_symmetry": float(base[i])})
    potential.append({"pair": i, "set_radius": 1, "pred_radius": float(do_radius[i]),
                      "set_concave": 1, "pred_concave": float(do_concave[i]),
                      "set_symmetry": 1, "pred_symmetry": float(base[i])})

def mechanistic(label, arm, outcome, criterion):
    return CausalClaim(kind="ate", outcome=outcome, treatment=arm,
                       arms=(do(arm, 1), do(arm, 0)), criterion=criterion,
                       assumptions=("the classifier is a deterministic mechanism",
                                    "do() assigns the feature for every unit "
                                    "(full compliance)"), label=label)

radius_claim = mechanistic("do_radius", "set_radius", "pred_radius", {"max": 0.0})
concave_claim = mechanistic("do_concave", "set_concave", "pred_concave",
                            {"min": -0.06, "max": 0.01})
placebo_claim = mechanistic("do_symmetry", "set_symmetry", "pred_symmetry",
                            {"min": -0.02, "max": 0.02})

mechanistic_session = CausalSession([radius_claim], table=potential)
paired = Strategy("paired", pair="pair")
oracle_paired = mechanistic_session.strategy(paired)
print("do(radius=17.2):", oracle_paired(radius_claim.id),
      oracle_paired.estimate(radius_claim.id))

candidate = mechanistic_session.candidate(
    strategy=paired, targets=[radius_claim], controls=[placebo_claim],
    tests=[(concave_claim, CERTIFIED)])
ev = mechanistic_session.evaluate(candidate, mechanistic_session.strategy(paired))
print("\ngates:", ", ".join(f"{g.name}={'PASS' if g.passed else 'FAIL'}"
                            for g in ev.gates))
print("census before:", ev.before, "-> after:", ev.after)
admission = mechanistic_session.promote(candidate,
                                        mechanistic_session.strategy(paired))
print("admission:", admission.approved, "-", admission.reason)

o = mechanistic_session.strategy(paired)
o(placebo_claim.id)
print("\nplacebo do(symmetry):", o.estimate(placebo_claim.id))
o2 = mechanistic_session.strategy(paired)
o2(concave_claim.id)
print("test do(concave=0.155):", o2.estimate(concave_claim.id))

do(radius=17.2): certified Estimate(-0.03891 ∈ [-0.07275, -0.005072], n=257, paired-difference(paired(pair)))

gates: labelled_tests=PASS, twin_holds=PASS, no_new_refuted=PASS, certified_gained=PASS, targeted_decline_moved=PASS
census before: {'certified': 0, 'declared': 0, 'declined': 1, 'refuted': 0} -> after: {'certified': 1, 'declared': 0, 'declined': 0, 'refuted': 0}
admission: True - admitted

placebo do(symmetry): Estimate(0 ∈ [0, 0], n=257, paired-difference(paired(pair)))
test do(concave=0.155): Estimate(-0.02335 ∈ [-0.05378, 0.007084], n=257, paired-difference(paired(pair)))


The model flips 3.9% of audit cases away from a malignant call when `radius` is
set to 17.2, with an interval entirely below zero. The placebo interval is
exactly `[0, 0]`: the model ignores `symmetry`, so the intervention cannot move
it. That is a stronger negative control than a statistical null.

## 4. Structure: bounded independence and a Berkson collider

With the full 569 rows and a declared MDE of 0.12, `texture` and `smoothness`
are **independent** (r ≈ −0.02). Conditioned on the diagnosis, they are
**dependent** (r ≈ −0.20): the diagnosis is a collider between two effects of
the same latent disease. The twin is the conditioned claim, which must be
`REFUTED`.

In [6]:
structure = [{"texture": float(X[i, FT]), "smooth": float(X[i, FSM]),
              "symmetry": float(X[i, FS]), "fractal": float(X[i, FF]),
              "malignant": float(y[i] == 0), "radius": float(X[i, FR])}
             for i in range(len(y))]

def independence(left, right, conditions=(), label=""):
    return CausalClaim(kind="independence", left=left, right=right,
                       conditions=conditions, criterion={"min": -1, "max": 1},
                       assumptions=("linear-Gaussian", "faithfulness"),
                       label=label)

ci_claims = [independence("texture", "smooth", label="tex/smooth"),
             independence("texture", "symmetry", label="tex/sym"),
             independence("texture", "fractal", label="tex/fractal"),
             independence("texture", "smooth", ("malignant",),
                          label="tex/smooth|malignant"),
             independence("texture", "symmetry", ("radius",),
                          label="tex/sym|radius")]

structure_session = CausalSession(ci_claims, table=structure)
ci = Strategy("ci", alpha=0.05, mde=0.12)
oracle_ci = structure_session.strategy(ci)
for claim in ci_claims:
    e = oracle_ci.estimate(claim.id) if oracle_ci(claim.id) else None
    print(f"{claim.label:22s} {oracle_ci.reasons[claim.id]:12s} "
          f"r={e.value:+.3f} p={e.detail['p_value']} "
          f"detectable_r={e.detail['detectable_r']}")

candidate = structure_session.candidate(
    strategy=ci, targets=ci_claims[:3],
    twin=[(ci_claims[3], REFUTED)],
    tests=[(ci_claims[4], CERTIFIED)])
admission = structure_session.promote(candidate, structure_session.strategy(ci))
print("\nadmission:", admission.approved, "-", admission.reason)

report = structure_session.markov_class(ci)
print("separated   :", report["separated"])
print("dependent   :", report["dependent"])
print("colliders   :", report["colliders"])
print("unidentified:", report["unidentified"])

tex/smooth             independent  r=-0.023 p=0.57785 detectable_r=0.1172
tex/sym                independent  r=+0.071 p=0.08883 detectable_r=0.1172
tex/fractal            independent  r=-0.076 p=0.06845 detectable_r=0.1172
tex/smooth|malignant   dependent    r=-0.203 p=0.0 detectable_r=0.1173
tex/sym|radius         independent  r=+0.006 p=0.8784 detectable_r=0.1173

admission: True - admitted
separated   : ['texture ⟂ smooth ∈ [-1, 1]', 'texture ⟂ symmetry ∈ [-1, 1]', 'texture ⟂ fractal ∈ [-1, 1]', 'texture ⟂ symmetry | radius ∈ [-1, 1]']
dependent   : ['texture ⟂ smooth | malignant ∈ [-1, 1]']
colliders   : [{'at': 'malignant', 'left': 'texture', 'right': 'smooth'}]
unidentified: ['orientation of non-collider edges', 'edges into colliders from outside the triple', 'anything not in the measured variables']


The report names the skeleton (non-adjacencies) and the collider, then states
explicitly that **edge orientation is not identified** from observational data.
There is no DAG and no edge score — an equivalence class, not a graph.

## 5. Tier 1: the audit stream as an interrupted time series

The audit is replayed as 32 batches. At batch 20 the policy tightens
(`τ: 0.50 → 0.99`) — a containment event. The outcome is the **hazard-release
rate**: the share of releases that carry an external hazard (a case with
`concave ≥ p60`), which is defined outside the model.

`series_from_journal(per="batch")` gives one point per batch; the estimator is
a segmented regression with the pre-trend as a gate. The placebo dates are
**claims**, not hidden checks.

In [7]:
def stream(seed=7, batches=32):
    order = list(range(len(y)))
    random.Random(seed).shuffle(order)
    hazard = X[:, FC] >= float(np.percentile(X[:, FC], 60))
    journal = Journal()
    per = len(order) // batches
    for b in range(batches):
        idx = order[b * per:(b + 1) * per]
        tau = 0.5 if b < 20 else 0.99
        for k, i in enumerate(idx):
            released = bool(p_benign[i] >= tau)
            journal.append({"kind": "decision", "batch": b,
                            "disposition": {
                                "verdict": "release" if released else "ask",
                                "certificate_id": f"{b}-{k}"},
                            "hazard_codes": ["high_concavity"]
                            if (released and hazard[i]) else []})
    journal.append({"kind": "containment", "containment_id": "c-1",
                    "reason": "hazard release", "at": "T"})
    return journal

journal = stream()
raw_series = series_from_journal(journal, metric="hazard_release", per="decision")
series = series_from_journal(journal, metric="hazard_release", per="batch")
print(f"journal verified: {journal.verify()} | per-decision {len(raw_series.points)}"
      f" | per-batch {len(series.points)} | metric {series.metric}")
print("batch rates:", [round(v, 2) for _, v in series.points])

def event_claim(event, criterion, label=""):
    return CausalClaim(kind="intervention_effect",
                       outcome="hazard_release_rate", event_index=event,
                       criterion=criterion,
                       assumptions=("no simultaneous shock",), label=label)

containment = event_claim(20, {"max": 0.0}, "containment")
placebo_12 = event_claim(12, {"min": -0.20, "max": 0.20}, "placebo_12")
placebo_8 = event_claim(8, {"min": -0.25, "max": 0.25}, "placebo_8")
its_session = CausalSession([containment, placebo_12, placebo_8], series=series)
its = Strategy("its", pre=14, post=8, placebo_offset=8)
oracle_its = its_session.strategy(its)
for claim in (containment, placebo_12, placebo_8):
    e = oracle_its.estimate(claim.id) if oracle_its(claim.id) else None
    print(f"{claim.label:12s} {oracle_its.reasons[claim.id]:12s} "
          f"est={e.value:+.3f} CI[{e.lo:+.3f}, {e.hi:+.3f}] "
          f"pre_slope_ci={e.detail['pre_slope_ci']} "
          f"placebo={e.detail['placebo_effect']}")

candidate = its_session.candidate(strategy=its, targets=[containment],
                                  controls=[placebo_12],
                                  tests=[(placebo_8, CERTIFIED)])
admission = its_session.promote(candidate, its_session.strategy(its))
print("\nadmission:", admission.approved, "-", admission.reason)
print("census:", its_session.status()["census"])

journal verified: True | per-decision 544 | per-batch 32 | metric hazard_release_rate
batch rates: [0.12, 0.18, 0.0, 0.06, 0.06, 0.12, 0.06, 0.12, 0.0, 0.0, 0.12, 0.24, 0.0, 0.06, 0.06, 0.12, 0.12, 0.12, 0.06, 0.12, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0]
containment  certified    est=-0.114 CI[-0.205, -0.022] pre_slope_ci=[-0.0037, 0.0105] placebo=-0.0279
placebo_12   certified    est=-0.028 CI[-0.148, +0.092] pre_slope_ci=[-0.0084, 0.0126] placebo=0.2118
placebo_8    certified    est=+0.008 CI[-0.183, +0.199] pre_slope_ci=[-0.0249, 0.0193] placebo=None

admission: True - admitted
census: {'certified': 1, 'declared': 0, 'declined': 2, 'refuted': 0}


The containment estimate is negative with an interval entirely below zero; the
pre-trend interval contains zero; the placebo-date claims certify the null (the
earlier placebo has a wider region because it has fewer pre-event batches — the
power cost of a short window is visible in the interval, not hidden).

## 6. The four numbers

No scalar anywhere: each audit reports `(certified, declared, declined,
refuted)`, the declared assumptions behind every certified claim, and the
placebo that could have refused it.

In [8]:
print("etiology audit :", session.status()["census"], "| contained then lifted")
print("mechanistic    :", mechanistic_session.status()["census"])
print("structure      :", structure_session.status()["census"])
print("event study    :", its_session.status()["census"])
print("\nassumptions certified (mechanistic):")
for a in mechanistic_session.assumptions():
    print("  -", a)
print("assumptions certified (structure):")
for a in structure_session.assumptions():
    print("  -", a)

etiology audit : {'certified': 1, 'declared': 0, 'declined': 1, 'refuted': 0} | contained then lifted
mechanistic    : {'certified': 1, 'declared': 0, 'declined': 0, 'refuted': 0}
structure      : {'certified': 3, 'declared': 0, 'declined': 2, 'refuted': 0}
event study    : {'certified': 1, 'declared': 0, 'declined': 2, 'refuted': 0}

assumptions certified (mechanistic):
  - do() assigns the feature for every unit (full compliance)
  - the classifier is a deterministic mechanism
assumptions certified (structure):
  - faithfulness
  - linear-Gaussian


## What this certifies — and what it refuses

`CERTIFIED` means *under the declared assumptions, the identified interval lies
inside the acceptance region*. It does not mean "true".

Refused on purpose: a DAG (observational data identify an equivalence class);
a `do()` on a manifestation (etiology is not identified here); over-adjustment
(it manufactures a refutation and is contained); positivity (no
extrapolation); power (bounded independence with a declared MDE); and any score
in place of the four numbers.

The extended explanation — identification, assumptions, the strategy gallery,
and the non-claims — is `causal_analysis.md`.